In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

DATASET_ROOT = Path("/root/Projet_Image/SH17dataset")
RUNS_DIR     = Path("/root/Projet_Image/runs")
BEST_WEIGHTS = RUNS_DIR / 'yolov8m_epi' / 'weights' / 'best.pt'
DEVICE       = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(str(BEST_WEIGHTS))
print(f"Modèle chargé : {BEST_WEIGHTS}")
print(f"Device        : {DEVICE}")

In [ ]:
results = model.val(
    data    = str(DATASET_ROOT / 'sh17.yaml'),
    split   = 'test',
    imgsz   = 640,
    batch   = 16,
    device  = DEVICE,
    verbose = False,
    plots   = False,
)

In [ ]:
map50     = float(results.box.map50)
map50_95  = float(results.box.map)
precision = float(results.box.mp)
recall    = float(results.box.mr)
f1_global = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"mAP50      : {map50:.3f}")
print(f"mAP50-95   : {map50_95:.3f}")
print(f"Précision  : {precision:.3f}")
print(f"Rappel     : {recall:.3f}")
print(f"F1 (global): {f1_global:.3f}")

In [ ]:
names       = model.names
class_idx   = results.box.ap_class_index
precision_c = results.box.p
recall_c    = results.box.r
f1_c        = results.box.f1
ap50_c      = results.box.ap50
ap_c        = results.box.ap

rows = [
    {
        'Classe':    names[i],
        'Précision': round(float(p), 3),
        'Rappel':    round(float(r), 3),
        'F1':        round(float(f), 3),
        'mAP50':     round(float(a50), 3),
        'mAP50-95':  round(float(a), 3),
    }
    for i, p, r, f, a50, a in zip(class_idx, precision_c, recall_c, f1_c, ap50_c, ap_c)
]
rows.append({
    'Classe':    'GLOBAL (moyenne)',
    'Précision': round(precision, 3),
    'Rappel':    round(recall, 3),
    'F1':        round(f1_global, 3),
    'mAP50':     round(map50, 3),
    'mAP50-95':  round(map50_95, 3),
})

table = pd.DataFrame(rows)
table

## Évaluation du modèle 1024×1024 sur le split test

Le modèle ci-dessus (`yolov8m_epi/weights/best.pt`, epoch 84, `imgsz=640`) reproduit le tableau de `documentation/04.Evaluation.md` section 2.3. Le modèle finalement retenu pour le déploiement est cependant celui réentraîné à `imgsz=1024` (`yolov8m_epi_1024-2/weights/best.pt`, epoch 79, voir `2_4_entraitement_1024.ipynb`). La section ci-dessous l'évalue à son tour sur le split test, à `imgsz=1024` (sa résolution d'entraînement), et reproduit le tableau de la section 2.4 (mAP50 = 0.697, mAP50-95 = 0.411, Précision = 0.801, Rappel = 0.618, F1 = 0.698).

In [ ]:
model_1024 = YOLO(str(RUNS_DIR / 'yolov8m_epi_1024-2' / 'weights' / 'best.pt'))

results_1024 = model_1024.val(
    data    = str(DATASET_ROOT / 'sh17.yaml'),
    split   = 'test',
    imgsz   = 1024,
    batch   = 8,
    device  = DEVICE,
    verbose = False,
    plots   = False,
)

In [ ]:
map50_1024     = float(results_1024.box.map50)
map50_95_1024  = float(results_1024.box.map)
precision_1024 = float(results_1024.box.mp)
recall_1024    = float(results_1024.box.mr)
f1_1024        = 2 * precision_1024 * recall_1024 / (precision_1024 + recall_1024) if (precision_1024 + recall_1024) > 0 else 0.0

print(f"mAP50      : {map50_1024:.3f}")
print(f"mAP50-95   : {map50_95_1024:.3f}")
print(f"Précision  : {precision_1024:.3f}")
print(f"Rappel     : {recall_1024:.3f}")
print(f"F1 (global): {f1_1024:.3f}")

In [ ]:
names_1024       = model_1024.names
class_idx_1024   = results_1024.box.ap_class_index
precision_c_1024 = results_1024.box.p
recall_c_1024    = results_1024.box.r
f1_c_1024        = results_1024.box.f1
ap50_c_1024      = results_1024.box.ap50
ap_c_1024        = results_1024.box.ap

rows_1024 = [
    {
        'Classe':    names_1024[i],
        'Précision': round(float(p), 3),
        'Rappel':    round(float(r), 3),
        'F1':        round(float(f), 3),
        'mAP50':     round(float(a50), 3),
        'mAP50-95':  round(float(a), 3),
    }
    for i, p, r, f, a50, a in zip(class_idx_1024, precision_c_1024, recall_c_1024, f1_c_1024, ap50_c_1024, ap_c_1024)
]
rows_1024.append({
    'Classe':    'GLOBAL (moyenne)',
    'Précision': round(precision_1024, 3),
    'Rappel':    round(recall_1024, 3),
    'F1':        round(f1_1024, 3),
    'mAP50':     round(map50_1024, 3),
    'mAP50-95':  round(map50_95_1024, 3),
})

table_1024 = pd.DataFrame(rows_1024)
table_1024